In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [3]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [4]:
def transform_exais_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [5]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [6]:
exais_dataset = pd.read_csv('../Dataset/ExAIS_SMS Spam Dataset/all_ExAIS_SPAM_Dataset.csv', na_values=["null", "NaN"], keep_default_na=True).fillna("")
exais_dataset.head()

,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,SMS,send,127,127,4/9/2014 8:29,1026,SPAM,CANCLE,,,...,,,,,,,,,,
1,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Your plan Smallie.is going to be renewed. Plea...,,,...,,,,,,,,,,
2,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Dear Glo subscriber,you just passed 80% of your Plan Validity Per...,,...,,,,,,,,,,
3,SMS,receive,127,127,1/9/2014 14:25,1026,SPAM,Welcome to Smallie. It will expire on 04/09/20...,,,...,,,,,,,,,,
4,SMS,send,127,127,25/08/2014 10:13,1026,SPAM,Info,,,...,,,,,,,,,,


In [7]:
exais_dataset.iloc[:, 7] = exais_dataset.iloc[:, 7:].astype(str).agg(" ".join, axis=1)

exais_dataset = exais_dataset.iloc[:, :8]

# exais_dataset.to_csv('../Dataset/exais_SMS Spam Dataset/cleaned_all_exais_SPAM_Dataset.csv')

exais_dataset.head(10)

,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message
0,SMS,send,127,127,4/9/2014 8:29,1026,SPAM,CANCLE
1,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Your plan Smallie.is going to be renewed. Plea...
2,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Dear Glo subscriber you just passed 80% of yo...
3,SMS,receive,127,127,1/9/2014 14:25,1026,SPAM,Welcome to Smallie. It will expire on 04/09/20...
4,SMS,send,127,127,25/08/2014 10:13,1026,SPAM,Info
5,SMS,send,127,127,25/08/2014 07:47,1026,SPAM,1
6,SMS,receive,127,127,25/08/2014 07:38,1026,SPAM,Please send :1 for Day Plans2 for Week Plans...
7,SMS,send,127,127,25/08/2014 07:38,1026,SPAM,ACTIVATE
8,SMS,receive,127,127,25/08/2014 07:37,1026,SPAM,Please send ACTIVATE to 127 for a list of opti...
9,SMS,send,127,127,25/08/2014 07:37,1026,SPAM,PAYYOU


In [8]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'exais Dataset_'+'.csv')['URL'].to_list())

In [9]:
exais_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'ExAis_Dataset_'+'.csv')['URL']
exais_dataset['Message Len'] = [len(i) for i in exais_dataset['Message']]
exais_dataset.head()

,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message,Extracted URL,Message Len
0,SMS,send,127,127,4/9/2014 8:29,1026,SPAM,CANCLE,NaN,26
1,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Your plan Smallie.is going to be renewed. Plea...,NaN,114
2,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Dear Glo subscriber you just passed 80% of yo...,NaN,104
3,SMS,receive,127,127,1/9/2014 14:25,1026,SPAM,Welcome to Smallie. It will expire on 04/09/20...,NaN,74
4,SMS,send,127,127,25/08/2014 10:13,1026,SPAM,Info,NaN,24


In [10]:
exais_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'ExAis Websites Analysis'+'.csv')
exais_website_analysis_data = exais_website_analysis_data.drop(columns=['ham', 'spam'])
exais_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://glo.lifestyle.com,glo.lifestyle.com,0,0,-1,0
1,www.redcrossnigeria.org,www.redcrossnigeria.org,75325,4803,200,0
2,http://whatsapp.com/dl/,whatsapp.com,261534,4199,200,0
3,www.mtnonline.com/about-mtn/contact-us,www.mtnonline.com,0,0,-1,0
4,http://www.gloworld.com/globackup/,www.gloworld.com,0,0,200,0


In [11]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [12]:
# for row in exais_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = exais_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(exais_website_analysis_data['FQDN'])}
website_data = exais_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
exais_dataset['FQDN'] = fqdn
exais_dataset['Website Size in KB'] = website_size
exais_dataset['Website Textual Content Length'] = text_content_len
exais_dataset['Status Code'] = status_code
exais_dataset['Parked'] = parked

In [16]:
exais_dataset = exais_dataset.replace('', np.nan)
exais_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_15224\3960323880.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exais_dataset = exais_dataset.replace('', np.nan)


,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,SMS,send,127,127,4/9/2014 8:29,1026,SPAM,CANCLE,NaN,26,NaN,NaN,NaN,NaN,NaN
1,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Your plan Smallie.is going to be renewed. Plea...,NaN,114,NaN,NaN,NaN,NaN,NaN
2,SMS,receive,127,127,4/9/2014 8:25,1026,SPAM,Dear Glo subscriber you just passed 80% of yo...,NaN,104,NaN,NaN,NaN,NaN,NaN
3,SMS,receive,127,127,1/9/2014 14:25,1026,SPAM,Welcome to Smallie. It will expire on 04/09/20...,NaN,74,NaN,NaN,NaN,NaN,NaN
4,SMS,send,127,127,25/08/2014 10:13,1026,SPAM,Info,NaN,24,NaN,NaN,NaN,NaN,NaN


In [17]:
# Counter(exais_dataset['FQDN'].to_list())
exais_dataset[(exais_dataset['Extracted URL'].notna()) & (exais_dataset['FQDN'].isna())]

,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(exais_dataset['FQDN'].to_list())
exais_dataset[(exais_dataset['Extracted URL'].notna()) & (exais_dataset['FQDN'].notna())]

,SMS,Occasion,Target1,Target2,Date,Unknown,Label,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
101,SMS,receive,63123,63123,28/05/2014 19:25,4315,SPAM,Egbetola Olanrewaju: who wants a free ipad? ht...,http://t.co/Qpe80nnaJi,215,t.co,3367.0,295.0,200.0,0.0
186,SMS,receive,Airtel,Airtel,2/1/2014 18:10,3128,HAM,Compliments of the season!! Enjoy free Interne...,http://g.co/freezone,174,g.co,2190.0,303.0,200.0,0.0
192,SMS,receive,Airtel,Airtel,26/11/2013 12:40,3128,HAM,Experience priceless and FAST internet on the ...,http://bit.ly/N1saY4,174,bit.ly,126176.0,8761.0,200.0,1.0
194,SMS,receive,Airtel,Airtel,14/11/2013 11:47,3128,HAM,If you still haven't experienced our incredibl...,http://bit.ly/1b2WYiu,172,bit.ly,126176.0,8761.0,200.0,1.0
203,SMS,receive,631,631,2/9/2014 14:06,4632,SPAM,Your friends have posted 480 updates this week...,https://fb.com/l/1NjTfumaF0i0P9M,152,fb.com,75842.0,645.0,200.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5183,SMS,receive,631,631,17/03/2014 03:53,75,SPAM,Kafayat you have 32 friend requests on Facebo...,https://fb.me/aRZ4jXUBNgri9B,97,fb.me,75837.0,645.0,200.0,0.0
5184,SMS,receive,631,631,9/3/2014 9:06,75,SPAM,Kafayat you have 32 friend requests on Facebo...,https://fb.me/25RlD8Yl1Dw8B9M,98,fb.me,75837.0,645.0,200.0,0.0
5220,SMS,receive,8181,8181,23/09/2013 20:56,3088,SPAM,Airtel Club 10: simeontawose : over 1000 math...,www.mathsvillage.org,139,www.mathsvillage.org,0.0,0.0,-1.0,0.0
5221,SMS,receive,8181,8181,22/09/2013 16:34,3088,SPAM,Airtel Club 10: diamond's hrt : I just saw a p...,www.facebook.com/airtelgroupy,156,www.facebook.com,75840.0,645.0,200.0,0.0


In [19]:
print(len(exais_dataset))

5240


In [20]:
#messages with URL
print(len(exais_dataset[(exais_dataset['Extracted URL'].notna())]), len(exais_dataset[(exais_dataset['Extracted URL'].notna())])/len(exais_dataset))

193 0.03683206106870229


In [24]:
#SPAM messages with URL
print(len(exais_dataset[(exais_dataset['Extracted URL'].notna()) & (exais_dataset['Label']=='SPAM')]), len(exais_dataset[(exais_dataset['Extracted URL'].notna()) & (exais_dataset['Label']=='SPAM')])/len(exais_dataset[exais_dataset['Label']=='SPAM']))

125 0.05319148936170213


In [ ]:
#unique FQDN
len(set(exais_dataset[(exais_dataset['FQDN'].notna())]['FQDN']))

64

In [27]:
only_unique_live_websites_data = exais_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites HAM
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['Label']=='HAM')]))

#live websites SPAM
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['Label']=='SPAM')]))

32
9
23


In [26]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites HAM
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['Label']=='HAM')]))

#parked websites SPAM
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['Label']=='SPAM')]))

10
4
6


In [ ]:
exais_dataset.to_csv('../Dataset/Refined_ExAIS_SMS_Spam_Dataset.csv', index=None)